# Manual Computing

## input

In [ ]:
import numpy as np

# 1. Define Demand and Technosphere (A) - Adjusted for connectivity
f = np.array([750, 0, 0, 0, 0, 0])

A = np.array([
    [1,        0,  0,   -0.01,0,    0   ], 
    [0,        1,  0,   0,    0,   -5   ], 
    [-1/750,   0,  1,   0,    0,    0   ], 
    [0,        0, -10,  1,    0,    0   ], 
    [0,        0, -0.8, 0,    1,    0   ], 
    [0,        0, -0.4, 0,    0,    1   ]  
])

# 2. Define Intervention Matrix (B) 
# Flows: Li-em, Heat, Coal, CO2, Li-ext, Water-ext, Water-em
B = np.array([
    [0, 0, 0.2, 0,   0,   4  ],  # Lithium emission: W
    [0, 0, 0.2, 0,   0,   0  ],  # Heat emission: W
    [0, 0, 0,   8, 0,   0  ],  # Coal resource: V
    [0, 0, 0,   24, 0,   0  ],  # CO2 emission: W
    [0, 0, 0,   0,   1, 0  ],  # Lithium extraction: V
    [0, 0, 0,   0,   1,   0  ],  # Water extraction: V
    [0, 0, 0,   0,   1,   0  ]   # Water emission: Er
])

B2 = np.array([
    [0, 0, 0.2, 0,   0,   4  ],  # Lithium emission: W
    [0, 0, 0.2, 0,   0,   0  ],  # Heat emission: W
    [0, 0, 0,   8, 0,   0  ],  # Coal resource: V
    [0, 0, 0,   24, 0,   0  ],  # CO2 emission: W
    [0, 0, 0,   0,   1, 0  ],  # Lithium extraction: V
    [0, 0, 0,   0,   1,   0  ],  # Water extraction: V
    [0, 0, 0,   0,   1,   0  ],   # Water emission: Er
    [0, 1, 0,   0,   0,   0  ]   # EoL LiBs: Ri
])

# 3. Categorization Dictionary
# flows names
flow_names = [
    "Lithium emission (kg)",
    "Heat emission (MJ)",
    "Coal resource (kg)",
    "CO2 emission (kg)",
    "Lithium resource (kg)",
    "Water resource (m3))",
    "Water emission (m3)",
    "EoL LiBs (kg)"
]

# Mapping flow index to the variable in your efficiency formulas
categorization = {
    0: 'W',  # Lithium emission
    1: 'W',  # Heat emission
    2: 'V',  # Coal resource
    3: 'W',  # CO2 emission
    4: 'V',  # Lithium extraction
    5: 'V',  # Water extraction
    6: 'Er', # Water emission
    7: 'Ri'  # Lithium EoL input
}

# 4. Units grouping for calculations
unit_groups = {
    "kg": [0, 2, 3, 4, 7],  # Li-em(0), Coal(2), CO2(3), Li-ext(4), EoL LiBs(7)
    "MJ": [1],               # Heat(1) - removed Coal from here as it's better in kg
    "m3": [5, 6]             # Water-ext(5), Water-em(6)
}

properties = {
    "MJ": {2: 9.9},      # Coal (index 2) converted from kg to MJ (1kg * 9.9 as in CED)
    "kg": {5: 1000.0, 6: 1000.0}, # Water (index 5,6) converted from m3 to kg (1m3 * 1000)
    "m3": {}              # Water already in m3, factor is 1.0
}

# 5. Run Calculation
s = np.linalg.solve(A, f)
lci = B @ s
lci2 = B2 @ s


## outputs

### scaling vector, and LCI results

In [ ]:
print("--- Traditional LCI (manual) ---")
print(f"Scaling Vector (s): {s}") # This shows which processes are 'active'
print("-" * 30)

for name, value in zip(flow_names, lci):
    print(f"{name}: {value:.5f}")

print("--- This works LCI (manual) ---")

for name, value in zip(flow_names, lci2):
    print(f"{name}: {value:.5f}")

### circularity indicators

In [ ]:
import numpy as np
import pandas as pd

# Run LCA calculations
s = np.linalg.solve(A, f)
lci_trad = B @ s
lci_dup = B2 @ s

# Define conversion properties for each unit
# Rr_FU = 750 MJ (only for MJ unit in duplication method, 0 otherwise)
Rr_FU_trad = {'kg': 0, 'MJ': 0, 'm3': 0}  # No Rr_FU for traditional
Rr_FU_dup = {'kg': 0, 'MJ': 750, 'm3': 0}  # Rr_FU only for duplication method

# Define calculations for each unit - FIXED WATER ASSIGNMENTS
units_config = {
    'kg': {
        'flows': [0, 2, 3, 4, 5, 6, 7],  # All flows except heat (1)
        'properties': {5: 1000.0, 6: 1000.0},  # Convert m3 to kg
        'V': [2, 4, 5],   # Coal, Li-ext, Water extraction (as resource)
        'W': [0, 3],   # Li-em, CO2 (as waste)
        'Er': [6],         # Water emission
        'Ri': [7],        # EoL LiBs
        'Rr': []          # No Rr flow in kg unit
    },
    'MJ': {
        'flows': [1, 2],   # Heat, Coal
        'properties': {2: 9.9},  # Convert kg Coal to MJ
        'V': [2],         # Coal (as energy)
        'W': [1],         # Heat (as waste energy)
        'Er': [],
        'Ri': [],
        'Rr': []
    },
    'm3': {
        'flows': [5, 6],   # Water-ext, Water-em
        'properties': {},  # Already in m3
        'V': [5],         # Water extraction (resource)
        'W': [],
        'Er': [6],        # Water emission (goes to recycling/environment)
        'Ri': [],
        'Rr': []
    }
}

# Function to calculate results for one LCI
def get_results(lci, method_name, rr_fu_dict):
    results = {}
    
    for unit, config in units_config.items():
        # print(f"\n--- {unit.upper()} calculation ---")
        
        # Calculate V, W, Er, Ri, Rr
        V = sum(lci[idx] * config['properties'].get(idx, 1.0) for idx in config['V'] if idx < len(lci))
        W = sum(lci[idx] * config['properties'].get(idx, 1.0) for idx in config['W'] if idx < len(lci))
        Er = sum(lci[idx] * config['properties'].get(idx, 1.0) for idx in config['Er'] if idx < len(lci))
        Ri = sum(lci[idx] * config['properties'].get(idx, 1.0) for idx in config['Ri'] if idx < len(lci))
        
        # Check if Rr exists in config
        if 'Rr' in config and config['Rr']:
            Rr = sum(lci[idx] * config['properties'].get(idx, 1.0) for idx in config['Rr'] if idx < len(lci))
        else:
            Rr = 0
        
        # Print details for verification
        # if unit == 'kg':
        #     print(f"  Coal (idx2): {lci[2] if 2 < len(lci) else 0:.4f} kg -> V")
        #     print(f"  Li-ext (idx4): {lci[4] if 4 < len(lci) else 0:.4f} kg -> V")
        #     print(f"  Water-ext (idx5): {lci[5] if 5 < len(lci) else 0:.4f} m3 -> {lci[5] * 1000:.2f} kg -> V")
        #     print(f"  Li-em (idx0): {lci[0] if 0 < len(lci) else 0:.4f} kg -> W")
        #     print(f"  CO2 (idx3): {lci[3] if 3 < len(lci) else 0:.4f} kg -> W")
        #     print(f"  Water-em (idx6): {lci[6] if 6 < len(lci) else 0:.4f} m3 -> {lci[6] * 1000:.2f} kg -> Er")
        
        # print(f"  V: {V:.4f}, W: {W:.4f}, Er: {Er:.4f}, Ri: {Ri:.4f}, Rr: {Rr:.4f}")
        
        # Calculate indicators
        denominator = V + Ri
        if denominator > 0:
            efficiency = (rr_fu_dict[unit] + Rr + Er) / denominator
            inefficiency = W / denominator
        else:
            efficiency, inefficiency = 0, 0
        
        results[unit] = {
            'Method': method_name,
            'V': V, 
            'W': W, 
            'Er': Er, 
            'Ri': Ri,
            'Rr': Rr,
            'Rr_FU': rr_fu_dict[unit],
            'Denominator': denominator,
            'Efficiency_η+': efficiency,
            'Inefficiency_η-': inefficiency
        }
    
    return pd.DataFrame(results).T

# Get results as DataFrames
trad_results = get_results(lci_trad, 'Traditional', Rr_FU_trad)
dup_results = get_results(lci_dup, 'Duplication', Rr_FU_dup)

# Create easy access DataFrames for each unit
trad_kg = trad_results.loc['kg'].to_frame().T
trad_MJ = trad_results.loc['MJ'].to_frame().T
trad_m3 = trad_results.loc['m3'].to_frame().T

dup_kg = dup_results.loc['kg'].to_frame().T
dup_MJ = dup_results.loc['MJ'].to_frame().T
dup_m3 = dup_results.loc['m3'].to_frame().T

# Print all results
print("\n" + "="*60)
print("TRADITIONAL METHOD")
print("="*60)
print(trad_results.round(3))

print("\n" + "="*60)
print("DUPLICATION METHOD")
print("="*60)
print(dup_results.round(3))

# Comparison table for Efficiency and Inefficiency
# print("\n" + "="*60)
# print("COMPARISON (η+ and η-)")
# print("="*60)
# comparison = pd.DataFrame({
#     'Trad_η+': [trad_results.loc['kg', 'Efficiency_η+'], trad_results.loc['MJ', 'Efficiency_η+'], trad_results.loc['m3', 'Efficiency_η+']],
#     'Trad_η-': [trad_results.loc['kg', 'Inefficiency_η-'], trad_results.loc['MJ', 'Inefficiency_η-'], trad_results.loc['m3', 'Inefficiency_η-']],
#     'Dup_η+': [dup_results.loc['kg', 'Efficiency_η+'], dup_results.loc['MJ', 'Efficiency_η+'], dup_results.loc['m3', 'Efficiency_η+']],
#     'Dup_η-': [dup_results.loc['kg', 'Inefficiency_η-'], dup_results.loc['MJ', 'Inefficiency_η-'], dup_results.loc['m3', 'Inefficiency_η-']]
# }, index=['kg', 'MJ', 'm3'])
# print(comparison.round(6))

In [ ]:
# Quick access examples
print(f"\nDuplication MJ efficiency: {dup_MJ['Efficiency_η+'].values[0]:.6f}")

# With brightway project

## init

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import bw2data as bd
import bw2calc as bc
import numpy as np
# local modules: circularity and lca
from circularity_lci.burden_free_analyzer import BurdenFreeAnalyzer
from circularity_lci.biosphere_flow_manager import BiosphereFlowManager
from circularity_lci.circularity_calculator import CircularityCalculator
# Set up project
project_name = "libs_circularity_example"
if project_name in bd.projects:
    bd.projects.delete_project(project_name, delete_dir=True)
bd.projects.set_current(project_name)


In [ ]:
excluded_flows = [
    "BOD5, Biological Oxygen Demand",
    "COD, Chemical Oxygen Demand",
    "DOC, Dissolved Organic Carbon",
    "TOC, Total Organic Carbon"
]

# valuable compartment for water flows, see function file for usability
valuable_water_compartments = ['water', 'undefined'] # for water flows
exclude_water = False  # Set to False if water flows are included # should be based on wet_mass or dry_mass => check dry_mass working
mass_strategy = "full_mass"      # options: "full_mass", "water_mass", "dry_mass"
energy_strategy = "full_energy"

# fit w<ith functions.py compliancies
database_provider = "_"  
database_version = "_"
database_systemmodel = "cutoff"
ecospold_folder = None


## make the lci

In [ ]:
# Create databases
technosphere_db_name = f"{database_provider}-{database_version}-{database_systemmodel}"
biosphere_db_name = f"{database_provider}-{database_version}-biosphere"

# Delete existing databases if they exist
if technosphere_db_name in bd.databases:
    del bd.databases[technosphere_db_name]
if biosphere_db_name in bd.databases:
    del bd.databases[biosphere_db_name]

# Create new databases
bd.Database(technosphere_db_name).write({})
bd.Database(biosphere_db_name).write({})

# Create biosphere flows
biosphere_db = bd.Database(biosphere_db_name)
biosphere_flows = [
    # Emissions
    ('Lithium', 'kilogram', ('soil', 'unspecified'), 'emission'),
    ('Heat', 'megajoule', ('air', 'unspecified'), 'emission'),
    ('Carbon dioxide', 'kilogram', ('air', 'unspecified'), 'emission'),
    ('Water', 'cubic meter', ('water', 'groundwater'), 'emission'),
    # Resources
    ('Coal, brown', 'kilogram', ('natural resource', 'in ground'), 'resource'),
    ('Lithium', 'kilogram', ('natural resource', 'in ground'), 'resource'),
    ('Water', 'cubic meter', ('natural resource', 'in ground'), 'resource'),
]

for name, unit, categories, flow_type in biosphere_flows:
    code = f"{name}_{categories[0]}_{categories[1]}"
    biosphere_db.new_node(
        code=code,
        name=name,
        unit=unit,
        categories=categories,
        type=bd.labels.biosphere_node_default,
    ).save()

# Create technosphere activities
technosphere_db = bd.Database(technosphere_db_name)

# 1. LiBs use
libs_use = technosphere_db.new_node(
    code='libs_use',
    name='LiBs use',
    unit='megajoule',
    location='FR',
    type=bd.labels.process_node_default,
)
libs_use.save()

# 2. LiBs EoL (cut-off)
libs_eol_cutoff = technosphere_db.new_node(
    code='libs_eol_cutoff',
    name='LiBs EoL (cut-off)',
    unit='kilogram',
    location='FR',
    type=bd.labels.process_node_default,
)
libs_eol_cutoff.save()

# 3. LiBs manufacturing
libs_manufacturing = technosphere_db.new_node(
    code='libs_manufacturing',
    name='LiBs manufacturing',
    unit='kilogram',
    location='FR',
    type=bd.labels.process_node_default,
)
libs_manufacturing.save()

# 4. Electricity production
electricity = technosphere_db.new_node(
    code='electricity_production',
    name='Electricity production',
    unit='kilowatt hour',
    location='FR',
    type=bd.labels.process_node_default,
)
electricity.save()

# 5. Lithium production
lithium_production = technosphere_db.new_node(
    code='lithium_production',
    name='Lithium production',
    unit='kilogram',
    location='FR',
    type=bd.labels.process_node_default,
)
lithium_production.save()

# 6. LiBs EoL Recycling
libs_recycling = technosphere_db.new_node(
    code='libs_recycling',
    name='LiBs EoL Recycling',
    unit='kilogram',
    location='FR',
    type=bd.labels.process_node_default,
)
libs_recycling.save()

# Add exchanges for each process
def get_flow(name, categories):
    return next(
        flow for flow in biosphere_db
        if flow['name'] == name and flow['categories'] == categories
    )

li_em = get_flow('Lithium', ('soil', 'unspecified'))
li_resource = get_flow('Lithium', ('natural resource', 'in ground'))
heat_em = get_flow('Heat', ('air', 'unspecified'))
coal = get_flow('Coal, brown', ('natural resource', 'in ground'))
co2 = get_flow('Carbon dioxide', ('air', 'unspecified'))
water_resource = get_flow('Water', ('natural resource', 'in ground'))
water_em = get_flow('Water', ('water', 'groundwater'))

# 1. LiBs use
libs_use.new_edge(
    input=libs_use,
    amount=1,
    type=bd.labels.production_edge_default,
).save()

# Consumes LiBs from manufacturing
libs_use.new_edge(
    input=libs_manufacturing,
    amount=1/750,
    type=bd.labels.consumption_edge_default,
).save()

# 2. LiBs EoL (cut-off)
libs_eol_cutoff.new_edge(
    input=libs_eol_cutoff,
    amount=1,
    type=bd.labels.production_edge_default,
).save()

# 3. LiBs manufacturing
libs_manufacturing.new_edge(
    input=libs_manufacturing,
    amount=1,
    type=bd.labels.production_edge_default,
).save()

# Consumes electricity
libs_manufacturing.new_edge(
    input=electricity,
    amount=10,
    type=bd.labels.consumption_edge_default,
).save()

# Consumes lithium from primary production
libs_manufacturing.new_edge(
    input=lithium_production,
    amount=0.8,
    type=bd.labels.consumption_edge_default,
).save()

# Consumes recycled lithium from EoL
libs_manufacturing.new_edge(
    input=libs_recycling,
    amount=0.4,
    type=bd.labels.consumption_edge_default,
).save()

# Emissions
libs_manufacturing.new_edge(
    input=li_em,
    amount=0.2,
    type=bd.labels.biosphere_edge_default,
).save()

libs_manufacturing.new_edge(
    input=heat_em,
    amount=0.2,
    type=bd.labels.biosphere_edge_default,
).save()

# 4. Electricity production
electricity.new_edge(
    input=electricity,
    amount=1,
    type=bd.labels.production_edge_default,
).save()

# Consumes coal
electricity.new_edge(
    input=coal,
    amount=8,
    type=bd.labels.biosphere_edge_default,
).save()

# Emits CO2
electricity.new_edge(
    input=co2,
    amount=24,
    type=bd.labels.biosphere_edge_default,
).save()

# Uses stored energy from LiBs
electricity.new_edge(
    input=libs_use,
    amount=0.01,
    type=bd.labels.consumption_edge_default,
).save()

# 5. Lithium production
lithium_production.new_edge(
    input=lithium_production,
    amount=1,
    type=bd.labels.production_edge_default,
).save()

# Extracts lithium resource
lithium_production.new_edge(
    input=li_resource,
    amount=1,
    type=bd.labels.biosphere_edge_default,
).save()

# Extracts water
lithium_production.new_edge(
    input=water_resource,
    amount=1,
    type=bd.labels.biosphere_edge_default,
).save()

# Emits water back
lithium_production.new_edge(
    input=water_em,
    amount=1,
    type=bd.labels.biosphere_edge_default,
).save()

# 6. LiBs EoL Recycling
libs_recycling.new_edge(
    input=libs_recycling,
    amount=1,
    type=bd.labels.production_edge_default,
).save()

# Consumes EoL LiBs from cut-off
libs_recycling.new_edge(
    input=libs_eol_cutoff,
    amount=5,
    type=bd.labels.consumption_edge_default,
).save()

# Emits lithium to ground
libs_recycling.new_edge(
    input=li_em,
    amount=4,
    type=bd.labels.biosphere_edge_default,
).save()

# Add properties to exchanges for mass/energy characterization
# Properties dict: {'wet mass': {'amount': value, 'unit': 'kilogram'}, 'energy content': {'amount': value, 'unit': 'megajoule'}}

# For Coal (resource flow)
coal_exc = next((exc for exc in electricity.exchanges() if exc.input == coal), None)
if coal_exc:
    coal_exc['properties'] = {
        'wet mass': {'amount': 1.0, 'unit': 'kilogram'},
        'energy content': {'amount': 9.9, 'unit': 'megajoule'}  # 9.9 megajoule per kilogram coal
    }
    coal_exc.save()

# For Water resource and emission flows
water_resource_exc  = next((exc for exc in lithium_production.exchanges() if exc.input == water_resource), None)
if water_resource_exc:
    water_resource_exc['properties'] = {
        'wet mass': {'amount': 1000, 'unit': 'kilogram'},  # 1 cubic meter = 1000 kilogram
        'water content': {'amount': 1.0, 'unit': 'kilogram/kilogram'}  # 100% water
    }
    water_resource_exc.save()

water_em_exc = next((exc for exc in lithium_production.exchanges() if exc.input == water_em), None)
if water_em_exc:
    water_em_exc['properties'] = {
        'wet mass': {'amount': 1000, 'unit': 'kilogram'},
        'water content': {'amount': 1.0, 'unit': 'kilogram/kilogram'}
    }
    water_em_exc.save()


print("Database created successfully!")

## circularity LCI

In [ ]:
burden_free_analyzer = BurdenFreeAnalyzer(
    project_name=project_name,
    database_provider=database_provider,
    ecospold_folder=ecospold_folder, 
    technosphere_db_name=technosphere_db_name,
    database_version=database_version,
    database_systemmodel=database_systemmodel
)
burden_free_analyzer.setup_project()

# --- 2. Analyze Burden-Free Activities ---
burden_free_activities = burden_free_analyzer.analyze_all_databases()

# --- 3. Initialize Biosphere Flow Manager ---
biosphere_flow_manager = BiosphereFlowManager(biosphere_db_name)  # Your biosphere DB

# --- 4. Process Burden-Free Activities (Create Biosphere Flows) ---
biosphere_flow_manager.process(burden_free_activities)

# Create calculation setup
setup_name = "libs_circularity_setup"
# Functional unit: 750 megajoule of stored energy
# libs_use_activity = bd.get_activity(("libs_technosphere", "libs_use"))
libs_use_activity = bd.get_activity((technosphere_db_name, "libs_use"))
# Create a fake LCIA method for CO2
method_name = ("Fake LCIA", "Climate Change", "CO2 only")
functional_unit = {libs_use_activity: 750}

# Get the CO2 biosphere flow
co2_flow = None
for flow in bd.Database(biosphere_db_name):
    if flow['name'] == 'Carbon dioxide':
        co2_flow = flow
        break

if co2_flow:
    # Create method with a fake characterization factor
    method_data = [(co2_flow.key, 1.0)]  # 1 kilogram CO2 = 1 point
    method_metadata = {
        'unit': 'kilogram CO2-eq',
        'description': 'Fake LCIA method for testing - only CO2 with CF = 1.0',
        'version': '1.0'
    }
    # Register method
    bd.Method(method_name).register(**method_metadata)
    bd.Method(method_name).write(method_data)
    print(f"✅ Created method: {method_name}")
    print(f" CO2 characterization factor: 1.0 {method_metadata['unit']}")
else:
    print("❌ CO2 flow not found")

bd.calculation_setups[setup_name] = {
    "inv": [functional_unit],  # Note the brackets - it needs to be a list
    "method": [("Fake LCIA", "Climate Change", "CO2 only")]
}
print(f"Calculation setup '{setup_name}' created with FU: 750 megajoule stored energy")

# Verify the setup
setup = bd.calculation_setups[setup_name]
print(f"\nSetup verification:")
print(f" Name: {setup_name}")
print(f" Functional unit: {setup['inv']}")

# Run a quick LCA to verify
lca = bc.LCA(functional_unit)
lca.lci()
print(f"\nLCA verification - Number of biosphere flows: {len(lca.biosphere_dict)}")
print(f"Total inventory sum: {lca.inventory.sum():.4f}")

# Now you can use your CircularityCalculator
circ_calc = CircularityCalculator(
    project_name=project_name,
    excluded_flows=excluded_flows,
    valuable_water_compartments=valuable_water_compartments,
    exclude_water=exclude_water,
    technosphere_db_name=technosphere_db_name,
    biosphere_db_name=biosphere_db_name,
    mass_strategy=mass_strategy,
    energy_strategy=energy_strategy
)

# Get inventory flows
flows_df = circ_calc.get_inventory_flows(setup_name)

# Compute circularity indicators
results_df, inefficiency, efficiency, detailed_flows = circ_calc.compute_circularity_efficiency_variables(
    flows_df,
    save_csv=True,
    setup_name=setup_name
)
print("\nCircularity Results:")
print(results_df.round(6))

In [ ]:
results_df

# 🔚